# Ridge Regression (L2 Regularization)

In this notebook we build a Ridge regression model to predict profit based on hours worked. We use polynomial features and normalization to improve model performance, and then use cross-validation to find the best regularization parameter (alpha).

In [1]:
import numpy as np
from sklearn.preprocessing import PolynomialFeatures
from sklearn.preprocessing import Normalizer
from sklearn.model_selection import train_test_split
from sklearn.linear_model import RidgeCV,Ridge
from sklearn.metrics import mean_absolute_error,root_mean_squared_error
from joblib import dump, load   

## Define the Dataset

We define a small dataset with 9 samples: `hours` (feature) and `profit` (target). Both are reshaped to column vectors as required by scikit-learn.

In [2]:
hours = np.array([1.5, 2, 3, 4, 5, 6, 7, 8, 10]).reshape(-1, 1)
profit = np.array([40, 50, 70, 90, 100, 110, 130, 145, 180]).reshape(-1, 1)

## Preprocessing, Training & Evaluation with a Fixed Alpha

1. **Polynomial Features (degree=4):** Expands the single `hours` feature into 4 features: hours, hours², hours³, hours⁴. This allows the linear model to capture non-linear relationships.
2. **Train/Test Split:** 80% training, 20% testing (random_state=101 for reproducibility).
3. **Normalization:** We fit the `Normalizer` on the training set only, then transform both train and test sets to avoid data leakage.
4. **Ridge Regression (alpha=5):** We train a Ridge model with a manually chosen alpha of 5 and evaluate using MAE and RMSE.

The high error values (MAE ≈ 36.4, RMSE ≈ 37.0) suggest that alpha=5 is too strong — the model is over-regularized.

In [3]:
poly_con = PolynomialFeatures(degree=4, include_bias=False)
norm_con = Normalizer()

poly_hours = poly_con.fit_transform(hours)
poly_hours_train, poly_hours_test, profit_train, profit_test = train_test_split(
    poly_hours, profit, test_size=0.2, random_state=101
)
norm_con.fit(poly_hours_train)
norm_poly_hours_train = norm_con.transform(poly_hours_train)
norm_poly_hours_test = norm_con.transform(poly_hours_test)

ridge = Ridge(alpha=5)
ridge.fit(norm_poly_hours_train, profit_train)
profit_predictions = ridge.predict(norm_poly_hours_test)
mae = mean_absolute_error(profit_test, profit_predictions)
rmse = root_mean_squared_error(profit_test, profit_predictions)
print(mae)
print(rmse)

36.366341417701555
36.98739144669071


## Cross-Validation to Find the Best Alpha

Instead of guessing alpha, we use `RidgeCV` which performs Leave-One-Out cross-validation over a range of alpha values (0.001 to ~0.9). The scoring metric is negative RMSE.

The result shows much lower errors (MAE ≈ 2.6, RMSE ≈ 2.7), confirming that the cross-validated alpha is far better than the manually chosen one.

In [4]:
alphas = np.arange(0.001, 1, 0.1)
ridge_cv = RidgeCV(alphas=alphas, scoring="neg_root_mean_squared_error")
ridge_cv.fit(norm_poly_hours_train, profit_train)
cv_profit_predictions = ridge_cv.predict(norm_poly_hours_test)
cv_mae = mean_absolute_error(profit_test, cv_profit_predictions)
cv_rmse = root_mean_squared_error(profit_test, cv_profit_predictions)
print(cv_mae)
print(cv_rmse)
alphas

2.6481629803539732
2.6712071817951326


array([0.001, 0.101, 0.201, 0.301, 0.401, 0.501, 0.601, 0.701, 0.801,
       0.901])

## Inspecting the Best Alpha and Coefficients

`ridge_cv.alpha_` returns the best alpha found by cross-validation (0.001), meaning very little regularization is needed for this dataset. `ridge_cv.coef_` shows the learned coefficients for each polynomial feature.

In [5]:
ridge_cv.alpha_

0.001

In [6]:
ridge_cv.coef_


array([ 105.25372038,  201.16315443, -526.82683956,  -63.18176258])

## Final Model: Retrain on All Data & Save

Now that we know the best alpha (0.001), we retrain a Ridge model on the **entire dataset** (no train/test split) to get the most accurate model possible. We then save the model, normalizer, and polynomial converter using `joblib.dump` so they can be reused later.

In [7]:
norm_poly_hours = norm_con.fit_transform(poly_hours)
final_ridge = Ridge(alpha=0.001)
final_ridge.fit(norm_poly_hours, profit)
dump(final_ridge,"final_ridge_model")
dump(norm_con,"normalizer")
dump(poly_con,"polynomial_converter")

['polynomial_converter']

## Load & Predict on New Data

We load the saved model and preprocessors, then predict the profit for a new input of 20 hours. The pipeline is: raw input → polynomial features → normalization → prediction. The model predicts a profit of ~178.6 for 20 hours.

In [8]:
norm = load("normalizer")
poly = load("polynomial_converter")
model = load("final_ridge_model")

new_data = np.array([20]).reshape(-1, 1)
poly_new_data = poly.transform(new_data)
norm_poly_new_data = norm.transform(poly_new_data)
preds = model.predict(norm_poly_new_data)
preds

array([178.62502385])